# CodeReview-Env — GRPO Training (Colab T4)

Train a code review agent using **Unsloth + HF TRL GRPOTrainer**.
Uses the live CodeReview-Env on HuggingFace Spaces for reward scoring.

### Prerequisites
- Runtime set to **T4 GPU** (Runtime > Change runtime type)
- Add `GROQ_API_KEY` and `HF_TOKEN` to the **Secrets** tab (🔑 icon on the left)

In [ ]:
# Cell 1: Install Dependencies (Optimized for Colab T4)
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

# Install Unsloth from source (pulls compatible transformers)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# Install TRL without heavy optional deps
!pip install --no-deps trl -q

# Install only what we actually need
!pip install peft accelerate bitsandbytes datasets requests matplotlib -q

# Remove packages that cause crashes (we don't use them)
!pip uninstall -y -q mergekit llm-blender vllm 2>/dev/null

In [ ]:
# Cell 2: Verify GPU
import torch
from unsloth import FastLanguageModel

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✓ Unsloth loaded! GPU: {gpu} ({vram:.1f} GB VRAM)")
else:
    print("❌ No GPU! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Cell 3: Configuration
# ─────────────────────────────────────────────────────────────────────
# Keys are loaded automatically from the Colab 🔑 Secrets tab.
# Make sure you have added GROQ_API_KEY and HF_TOKEN there!
# ─────────────────────────────────────────────────────────────────────
import os
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    os.environ["HF_TOKEN"]     = userdata.get("HF_TOKEN")
    print("✓ Secrets loaded from Colab Secrets tab")
except ImportError:
    print("ℹ Running locally — ensure .env is set")

# Environment server
os.environ["ENV_URL"]      = "https://dharaneswarreddy-codereview-env.hf.space"

# Student model
os.environ["TRAIN_MODEL"]  = "Qwen/Qwen2.5-1.5B-Instruct"

# Where to push the trained model (requires HF_TOKEN with write access)
os.environ["SAVE_REPO"]    = "DharaneswarReddy/codereview-agent"

# ╔═══════════════════════════════════════════════════╗
# ║  Set to 10 for a quick test, 300 for a full run  ║
os.environ["NUM_TRAIN_STEPS"] = "300"
# ╚═══════════════════════════════════════════════════╝

print(f"  ENV_URL:          {os.environ.get('ENV_URL')}")
print(f"  TRAIN_MODEL:      {os.environ.get('TRAIN_MODEL')}")
print(f"  SAVE_REPO:        {os.environ.get('SAVE_REPO')}")
print(f"  NUM_TRAIN_STEPS:  {os.environ.get('NUM_TRAIN_STEPS')}")

In [ ]:
# Cell 4: Clone Repository & Start Training
import os

!git clone https://github.com/GojoV339/MetaXScaler_Hackathon.git codereview-env 2>/dev/null || true
# Always pull latest code so train_grpo.py has the JSONL logging
!cd codereview-env && git pull
%cd codereview-env

# Clear any stale log from a previous run
if os.path.exists("training_log.jsonl"):
    os.remove("training_log.jsonl")
    print("✓ Cleared old training_log.jsonl")

# Run the GRPO training script
exec(open("training/train_grpo.py").read())


In [ ]:
# Cell 5: Training Analytics Dashboard (6 plots)
import json, os, numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Make sure we are in the right directory
if not os.path.exists("training_log.jsonl"):
    try:
        os.chdir("/content/codereview-env")
    except Exception:
        pass

log_path = "training_log.jsonl"
if not os.path.exists(log_path):
    print("❌ training_log.jsonl not found!")
    print(f"   CWD: {os.getcwd()}")
    print("   Try: !ls -la  to see what files exist")
else:
    rows = [json.loads(l) for l in open(log_path)]
    steps     = [r["step"]             for r in rows]
    rewards   = [r["mean_reward"]       for r in rows]
    levels    = [r["curriculum_level"]  for r in rows]
    hacks     = [r["hack_count"]         for r in rows]
    comp_keys = list(rows[0]["components"].keys())
    comp_data = {k: [r["components"][k] for r in rows] for k in comp_keys}

    def rolling(arr, w=10):
        return [np.mean(arr[max(0,i-w):i+1]) for i in range(len(arr))]

    plt.style.use("dark_background")
    fig = plt.figure(figsize=(20, 22))
    fig.suptitle("CodeReview-Agent â GRPO Training Dashboard", fontsize=22, fontweight="bold", color="white", y=0.98)
    gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)
    palette = ["#60A5FA", "#34D399", "#F59E0B", "#F87171", "#A78BFA"]

    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(steps, rewards, color="#94A3B8", lw=1.2, alpha=0.5, label="Per-step")
    ax1.plot(steps, rolling(rewards, 15), color="#60A5FA", lw=2.5, label="Rolling avg (15)")
    ax1.axhline(0.45, color="#34D399", ls="--", lw=1.5, label="Curriculum threshold")
    ax1.set_title("â  Overall Mean Reward", fontsize=14, fontweight="bold")
    ax1.set_xlabel("Training Step"); ax1.set_ylabel("Reward (0â1)")
    ax1.set_ylim(0, 1); ax1.legend(fontsize=9); ax1.grid(alpha=0.2)

    ax2 = fig.add_subplot(gs[0, 1])
    smth = [rolling(comp_data[k], 10) for k in comp_keys]
    ax2.stackplot(steps, smth, labels=comp_keys, colors=palette, alpha=0.85)
    ax2.set_title("â¡ Reward Components (Stacked)", fontsize=14, fontweight="bold")
    ax2.set_xlabel("Training Step"); ax2.set_ylabel("Component Value")
    ax2.set_ylim(0, 1); ax2.legend(loc="upper left", fontsize=9); ax2.grid(alpha=0.2)

    ax3 = fig.add_subplot(gs[1, 0])
    for i, k in enumerate(comp_keys):
        ax3.plot(steps, rolling(comp_data[k], 10), color=palette[i], lw=2, label=k)
    ax3.set_title("â¢ Individual Component Trends", fontsize=14, fontweight="bold")
    ax3.set_xlabel("Training Step"); ax3.set_ylabel("Score (0â1)")
    ax3.set_ylim(0, 1); ax3.legend(fontsize=9); ax3.grid(alpha=0.2)

    ax4 = fig.add_subplot(gs[1, 1])
    ax4.step(steps, levels, where="post", color="#F59E0B", lw=2.5)
    ax4.fill_between(steps, levels, step="post", alpha=0.15, color="#F59E0B")
    ax4.set_title("â£ Curriculum Level Progression", fontsize=14, fontweight="bold")
    ax4.set_xlabel("Training Step"); ax4.set_ylabel("Curriculum Level")
    ax4.set_yticks([1, 2, 3]); ax4.set_yticklabels(["Lv1 Detection", "Lv2 Classification", "Lv3 Full Review"])
    ax4.grid(alpha=0.2)

    ax5 = fig.add_subplot(gs[2, 0])
    hack_deltas = [hacks[0]] + [hacks[i]-hacks[i-1] for i in range(1, len(hacks))]
    colors_h = ["#F87171" if d > 0 else "#94A3B8" for d in hack_deltas]
    ax5.bar(steps, hack_deltas, color=colors_h, width=0.8, alpha=0.9)
    ax5.plot(steps, rolling(hack_deltas, 10), color="#FBBF24", lw=2, label="Rolling avg")
    ax5.set_title("â¤ Reward Hacking Events per Step", fontsize=14, fontweight="bold")
    ax5.set_xlabel("Training Step"); ax5.set_ylabel("New Hack Events")
    ax5.legend(fontsize=9); ax5.grid(alpha=0.2)

    ax6 = fig.add_subplot(gs[2, 1])
    mid = len(rewards) // 2
    ax6.hist(rewards[:mid], bins=20, color="#94A3B8", alpha=0.7, label="First half")
    ax6.hist(rewards[mid:], bins=20, color="#34D399", alpha=0.7, label="Second half")
    ax6.axvline(np.mean(rewards[:mid]), color="#94A3B8", ls="--", lw=2)
    ax6.axvline(np.mean(rewards[mid:]), color="#34D399", ls="--", lw=2)
    ax6.set_title("â¥ Reward Distribution: Before vs After", fontsize=14, fontweight="bold")
    ax6.set_xlabel("Reward Score"); ax6.set_ylabel("Frequency")
    ax6.legend(fontsize=9); ax6.grid(alpha=0.2)

    plt.savefig("training_dashboard.png", dpi=150, bbox_inches="tight", facecolor="#0F172A")
    plt.show()
    print(f"\nâ Final mean reward : {np.mean(rewards[-20:]):.3f}")
    print(f"  Peak reward       : {max(rewards):.3f}")
    print(f"  Total hack events : {hacks[-1]}")
    print(f"  Final curriculum  : Level {levels[-1]}")


In [ ]:
# Cell 6: Per-Component Summary Table
import json, os, numpy as np

log_path = "training_log.jsonl"
if not os.path.exists(log_path):
    print("❌ training_log.jsonl not found — run Cell 4 first!")
else:
    rows = [json.loads(l) for l in open(log_path)]
    comp_keys = list(rows[0]["components"].keys())
    weights = {"format": 0.15, "detection": 0.30, "classification": 0.20, "confidence": 0.15, "quality": 0.20}

    print("\n" + "="*65)
    print(f"  {'Component':<18} {'Weight':>7} {'Start Avg':>11} {'End Avg':>9} {'Change':>8}")
    print("="*65)
    n = max(5, len(rows)//5)
    for k in comp_keys:
        vals = [r["components"][k] for r in rows]
        s = np.mean(vals[:n])
        e = np.mean(vals[-n:])
        arrow = "▲" if e > s else "▼"
        print(f"  {k:<18} {weights.get(k,0)*100:>5.0f}%  {s:>10.3f}  {e:>9.3f}  {arrow} {e-s:+.3f}")
    print("="*65)
    all_rewards = [r["mean_reward"] for r in rows]
    print(f"  {'TOTAL REWARD':<18}         {np.mean(all_rewards[:n]):>10.3f}  {np.mean(all_rewards[-n:]):>9.3f}  {'▲' if np.mean(all_rewards[-n:])>np.mean(all_rewards[:n]) else '▼'} {np.mean(all_rewards[-n:])-np.mean(all_rewards[:n]):+.3f}")
    print("="*65)

In [ ]:
# Cell 7: Teacher Model Baseline Test (llama-3.3-70b via Groq)
from openai import OpenAI
import requests, os

groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY"),
)

ENV_URL = os.environ["ENV_URL"]
test_obs = requests.post(f"{ENV_URL}/reset", json={"task_level": 1}).json()

groq_resp = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a senior code reviewer. Return valid JSON."},
        {"role": "user",   "content": f"Review this code:\n```\n{test_obs['code']}\n```\nReturn JSON: {{\"has_bug\": bool, \"bug_type\": str, \"severity\": str, \"suggested_fix\": str}}"}
    ],
    max_tokens=256,
    temperature=0.6,
)
print(f"Teacher Model (llama-3.3-70b) response:\n{groq_resp.choices[0].message.content}")